# Beyond-accuracy analysis: COSETTE/MARIUS vs SASRec++

Item-side distribution metrics (catalog coverage, Gini, normalized Shannon entropy, average recommendation popularity, APLT, novelty, intra-list diversity, MARIUS hallucination rate, popularity-decile exposure, and tail recall) that the base paper (arXiv:2508.14910) only shows informally (Fig 4 decile plot, Fig 8 collisions) but never tabulates.

Metrics are defined in `scripts/extensions/beyond_accuracy.py`. They run on the ranked Top-K lists that `sasrec.search` / `marius.search` already produce, so nothing in the authors' model or eval code changes.

**Data note.** Real numbers need the Top-K dumps produced on Snellius by `jobs/22_dump_topk_{beauty,sports}.sbatch` (a single read-only eval pass per model/seed). Part A below runs with no data so you can verify the metric implementations locally; Part B activates automatically once the dumps are under `reports/extensions/topk/<category>/`.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'scripts' / 'extensions' / 'beyond_accuracy.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from scripts.extensions import beyond_accuracy as ba

DUMP_ROOT = REPO / 'reports' / 'extensions' / 'topk'
SEEDS = [42, 43, 44, 45, 46]
CATEGORIES = {'Beauty': 'beauty', 'Sports_and_Outdoors': 'sports'}
print('repo:', REPO)

repo: /home/stas/Desktop/University/RECSYS/rec_sys_cosette_marius


## Part A. Synthetic sanity demo (no data required)
Three toy recommenders over a Zipfian catalog: a popularity-chaser, a uniform one, and a generative one with injected hallucinations. The metrics should order them as expected (chaser = high Gini, low coverage, low novelty, low APLT).

In [2]:
rng = np.random.default_rng(0)
n_catalog, n_users, k = 1000, 2000, 10
item_pop = ((1.0 / np.arange(1, n_catalog + 1)) * 1e5).astype(np.int64) + 1
p = item_pop / item_pop.sum()
item_emb = rng.normal(size=(n_catalog, 32))
head50 = np.argsort(-item_pop)[:50]

recs = {
    'popularity-chaser': [list(rng.choice(head50, k, replace=False)) for _ in range(n_users)],
    'uniform':           [list(rng.choice(n_catalog, k, replace=False)) for _ in range(n_users)],
    'generative':        [[(-1 if rng.random() < 0.08 else int(x))
                            for x in rng.choice(n_catalog, k, replace=False, p=p)] for _ in range(n_users)],
}
demo = {name: ba.compute_all(r, item_pop, n_catalog, item_emb=item_emb, k=k) for name, r in recs.items()}
pd.DataFrame(demo).T[['coverage','gini','entropy_norm','arp','aplt','novelty','ild','hallucination_rate']].round(4)

,coverage,gini,entropy_norm,arp,aplt,novelty,ild,hallucination_rate
popularity-chaser,0.050,0.9512,0.5662,8865.928,0.0000,7.1997,0.9981,0.0000
uniform,1.000,0.1231,0.9965,742.695,0.7992,11.4330,1.0000,0.0000
generative,0.983,0.7224,0.7916,15774.739,0.2401,7.9570,1.0020,0.0819


## Part B. Real data (activates after the Snellius dump)
Loads the dumped Top-K and support tables, converts to catalog item indices (SASRec: vocab id minus special-token offset; MARIUS: semantic-ID tuple lookup, with unmatched tuples marked as hallucinations), and computes the suite per model and seed.

In [3]:
# Single source of truth: the SASRec vocab-offset and MARIUS semantic-ID
# token de-offset both live in compute_beyond_accuracy.py so the notebook and
# the headless CLI can never drift. Thin wrappers adapt category -> dump_dir.
from scripts.extensions.compute_beyond_accuracy import (
    load_support as _load_support, load_model_recs as _load_model_recs)

def load_support(category):
    return _load_support(DUMP_ROOT / category)

def load_model_recs(category, model, seed, meta, t2i):
    return _load_model_recs(DUMP_ROOT / category, model, seed, meta, t2i)

available = {c: (DUMP_ROOT / c / 'meta.json').exists() for c in CATEGORIES}
print('dumps present:', available)
if not any(available.values()):
    print('\nNo dumps yet. On Snellius run, per seed:')
    print('  sbatch --export=ALL,SEED=42 jobs/22_dump_topk_beauty.sbatch')
    print('  sbatch --export=ALL,SEED=42 jobs/22_dump_topk_sports.sbatch')
    print('then copy reports/extensions/topk/ back here.')

dumps present: {'Beauty': True, 'Sports_and_Outdoors': True}


In [4]:
def beyond_accuracy_table(category, ks=(10, 20)):
    meta, pop, emb, t2i = load_support(category)
    rows = []
    for model in ('sasrec', 'marius'):
        for seed in SEEDS:
            if not (DUMP_ROOT / category / f'{model}_seed{seed}_topk.npz').exists():
                continue
            recs, targets = load_model_recs(category, model, seed, meta, t2i)
            for k in ks:
                m = ba.compute_all(recs, pop, meta['n_catalog'], item_emb=emb, targets=targets, k=k)
                m.update({'category': category, 'model': model, 'seed': seed})
                rows.append(m)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    metric_cols = [c for c in df.columns if c not in ('category','model','seed','k')]
    return df.groupby(['category','model','k'])[metric_cols].mean().round(4)

for c, present in available.items():
    if present:
        display(beyond_accuracy_table(c))

coverage    gini  entropy_norm      arp    aplt  novelty  \
category model  k                                                              
Beauty   marius 10    0.7073  0.7943        0.8472  55.9012  0.2793  12.2870   
                20    0.8306  0.7480        0.8710  50.5444  0.3125  12.4355   
         sasrec 10    0.8130  0.8115        0.8144  74.3011  0.2278  11.9114   
                20    0.8970  0.7953        0.8312  67.4059  0.2258  11.9926   

                    hallucination_rate  recall  tail_recall  
category model  k                                            
Beauty   marius 10              0.0023  0.0826       0.0366  
                20              0.0034  0.1227       0.0607  
         sasrec 10              0.0000  0.0904       0.0454  
                20              0.0000  0.1259       0.0613

coverage    gini  entropy_norm       arp  \
category            model  k                                              
Sports_and_Outdoors marius 10    0.3924  0.9134        0.7669  101.4302   
                           20    0.5385  0.8792        0.8012   89.0486   
                    sasrec 10    0.7372  0.8706        0.7648  129.1476   
                           20    0.8354  0.8579        0.7862  112.4150   

                                 aplt  novelty  hallucination_rate  recall  \
category            model  k                                                 
Sports_and_Outdoors marius 10  0.1104  11.9784              0.0000  0.0471   
                           20  0.1481  12.1993              0.0001  0.0739   
                    sasrec 10  0.1612  11.8295              0.0000  0.0506   
                           20  0.1592  11.9504              0.0000  0.0747   

                               tail_recall  
category            model  k                
Sports_and_Outdoors marius 10       0.0112  
                           20       0.0212  
                    sasrec 10       0.0215  
                           20       0.0293

In [5]:
# Popularity-decile exposure profile (absolute version of the paper's Fig 4).
def decile_table(category, k=10):
    meta, pop, emb, t2i = load_support(category)
    out = {}
    for model in ('sasrec', 'marius'):
        seeds = [s for s in SEEDS if (DUMP_ROOT / category / f'{model}_seed{s}_topk.npz').exists()]
        if not seeds:
            continue
        profiles = []
        for s in seeds:
            recs, _ = load_model_recs(category, model, s, meta, t2i)
            profiles.append(ba.popularity_decile_exposure(recs, pop, k=k))
        out[model] = np.mean(profiles, axis=0)
    if not out:
        return pd.DataFrame()
    return pd.DataFrame(out, index=[f'D{i+1}' for i in range(len(next(iter(out.values()))))]).round(4)

for c, present in available.items():
    if present:
        print(c, '- exposure share per popularity decile (D1=rarest, D10=most popular)')
        display(decile_table(c))

Beauty - exposure share per popularity decile (D1=rarest, D10=most popular)


,sasrec,marius
D1,0.0146,0.0055
D2,0.0155,0.0120
D3,0.0185,0.0174
D4,0.0193,0.0193
D5,0.0246,0.0280
D6,0.0322,0.0416
D7,0.0408,0.0603
D8,0.0621,0.0940
D9,0.1245,0.1785
D10,0.6478,0.5435


Sports_and_Outdoors - exposure share per popularity decile (D1=rarest, D10=most popular)


,sasrec,marius
D1,0.0096,0.0007
D2,0.0108,0.0018
D3,0.0128,0.0035
D4,0.0128,0.0042
D5,0.0170,0.0087
D6,0.0233,0.0166
D7,0.0295,0.0245
D8,0.0456,0.0513
D9,0.0895,0.1186
D10,0.7493,0.7700


## What to read off Part B
- **Coverage / Gini / entropy / novelty / APLT / ILD:** does the generative model (MARIUS) spread exposure wider and surface more tail items than discriminative SASRec++, or the reverse? The paper hints (Fig 4) that MARIUS shifts toward mid-popular items; this quantifies it.
- **Tail recall:** does any diversity gain come with correct tail predictions, or just noise? (accuracy-meets-beyond-accuracy)
- **Hallucination rate:** MARIUS only; the share of generated tuples mapping to no item.
- **Decile profile:** the absolute exposure distribution; compare against the paper's Fig 4 difference plot.

Open scientific hook (needs a content-only RQ-VAE tokenizer variant): is COSETTE's *collaborative* tokenization itself a diversity/popularity-bias lever, vs content-only semantic IDs? Differentiate from Ghost (arXiv:2605.16825) and CRAB (arXiv:2604.05113), which target TIGER/LLM generative recommenders, not MARIUS.